# 11 — Streaming Halt Deep Dive: Token-Level Oversight

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/11_streaming_halt_deep_dive.ipynb)

Director-AI's streaming kernel monitors coherence **token by token**
and halts generation the instant quality degrades. Three independent
halt mechanisms work in parallel.

**What you'll learn:**
1. Three halt mechanisms and when each fires
2. Tuning halt parameters for your domain
3. Async streaming for WebSocket production
4. Visualising coherence timelines
5. False-halt prevention

In [ ]:
!pip install -q director-ai

## 1. The Three Halt Mechanisms

| Mechanism | Trigger | Purpose |
|-----------|---------|--------|
| **Hard limit** | Single token below floor | Catches catastrophic drops |
| **Sliding window** | Rolling average below threshold | Detects sustained decay |
| **Downward trend** | Coherence drops by Δ over N tokens | Catches gradual drift |

In [ ]:
from director_ai.core import StreamingKernel


def run_stream(kernel, tokens, scores):
    """Run a synthetic stream and print events."""
    idx = 0

    def score_fn(tok):
        nonlocal idx
        s = scores[min(idx, len(scores) - 1)]
        idx += 1
        return s

    session = kernel.stream_tokens(iter(tokens), score_fn)

    for ev in session.events:
        marker = f" <<<HALT ({session.halt_reason})" if ev.halted else ""
        print(f"  [{ev.index:2d}] {ev.coherence:.3f}  {ev.token!r}{marker}")

    return session

In [ ]:
# Scenario 1: Hard limit halt — single catastrophic token
print("=== Hard Limit Halt ===")
kernel = StreamingKernel(hard_limit=0.3, window_size=5, window_threshold=0.4)

tokens = ["The", " capital", " of", " France", " is", " actually", " Mars", "."]
scores = [0.95, 0.92, 0.93, 0.90, 0.88, 0.50, 0.15, 0.10]

session = run_stream(kernel, tokens, scores)
print(f"\nHalted: {session.halted}, tokens emitted: {session.token_count}/{len(tokens)}")

In [ ]:
# Scenario 2: Sliding window halt — gradual decay
print("=== Sliding Window Halt ===")
kernel = StreamingKernel(hard_limit=0.2, window_size=4, window_threshold=0.5)

tokens = ["Water", " boils", " at", " 100", "°C", " but", " some", " say", " it", " is", " 50", "°C"]
scores = [0.92, 0.90, 0.91, 0.88, 0.85, 0.65, 0.55, 0.48, 0.42, 0.38, 0.30, 0.25]

session = run_stream(kernel, tokens, scores)
print(f"\nHalted: {session.halted}, tokens emitted: {session.token_count}/{len(tokens)}")

In [ ]:
# Scenario 3: Downward trend halt
print("=== Downward Trend Halt ===")
kernel = StreamingKernel(
    hard_limit=0.1,
    window_size=5,
    window_threshold=0.3,
    trend_window=5,
    trend_threshold=0.3,  # halt if drop > 0.3 over 5 tokens
)

tokens = ["The", " answer", " is", " approximately", " well", " actually", " nobody", " knows"]
scores = [0.90, 0.85, 0.78, 0.70, 0.62, 0.55, 0.50, 0.45]

session = run_stream(kernel, tokens, scores)
print(f"\nHalted: {session.halted}, tokens emitted: {session.token_count}/{len(tokens)}")

## 2. Tuning for Your Domain

Different domains need different sensitivity.

In [ ]:
from director_ai.core.config import DirectorConfig

# Medical: aggressive halting — false negatives are dangerous
medical = DirectorConfig.from_profile("medical")
medical_kernel = StreamingKernel(
    hard_limit=medical.hard_limit,
    window_size=3,
    window_threshold=medical.coherence_threshold,
)

# Creative: relaxed — hallucination is a feature
creative = DirectorConfig.from_profile("creative")
creative_kernel = StreamingKernel(
    hard_limit=creative.hard_limit,
    window_size=8,
    window_threshold=creative.coherence_threshold,
)

print(f"Medical:  hard_limit={medical.hard_limit}, threshold={medical.coherence_threshold}")
print(f"Creative: hard_limit={creative.hard_limit}, threshold={creative.coherence_threshold}")

# Same stream, different outcomes
tokens = ["Take", " 200", "mg", " or", " maybe", " 2000", "mg"]
scores = [0.95, 0.90, 0.85, 0.60, 0.50, 0.35, 0.20]

print("\n--- Medical kernel ---")
s1 = run_stream(medical_kernel, tokens, scores)

print("\n--- Creative kernel ---")
s2 = run_stream(creative_kernel, tokens, scores)

print(f"\nMedical halted at token {s1.token_count}, Creative halted at token {s2.token_count}")

## 3. Async Streaming for WebSocket Production

The `AsyncStreamingKernel` is designed for WebSocket servers.

```python
from director_ai.core import AsyncStreamingKernel

kernel = AsyncStreamingKernel(hard_limit=0.4, window_size=5)

async def ws_handler(websocket):
    token_gen = llm.stream(prompt)
    
    async for event in kernel.stream_tokens(token_gen, score_fn):
        await websocket.send_json({
            "token": event.token,
            "coherence": event.coherence,
            "halted": event.halted,
        })
        if event.halted:
            await websocket.send_json({"halt_reason": event.halt_reason})
            break
```

In [ ]:
import asyncio
from director_ai.core import AsyncStreamingKernel


async def demo_async_stream():
    kernel = AsyncStreamingKernel(hard_limit=0.3, window_size=4, window_threshold=0.4)

    tokens = ["Paris", " is", " the", " capital", " of", " Jupiter", "."]
    scores = [0.95, 0.93, 0.92, 0.90, 0.85, 0.20, 0.10]
    idx = 0

    async def token_gen():
        for t in tokens:
            yield t

    def score_fn(tok):
        nonlocal idx
        s = scores[min(idx, len(scores) - 1)]
        idx += 1
        return s

    events = []
    async for event in kernel.stream_tokens(token_gen(), score_fn):
        events.append(event)
        status = " HALT" if event.halted else ""
        print(f"  [{event.index}] {event.coherence:.3f} {event.token!r}{status}")
        if event.halted:
            break

    print(f"\nTotal events: {len(events)}")


await demo_async_stream()

## 4. Visualising Coherence Timelines

In [ ]:
def ascii_timeline(events, hard_limit, window_threshold):
    """ASCII art coherence timeline."""
    width = 50
    print(f"{'Token':<12} {'Score':>5}  {'':─<{width}}")

    for ev in events:
        bar_len = int(ev.coherence * width)
        bar = "█" * bar_len + "░" * (width - bar_len)

        # Markers for thresholds
        hard_pos = int(hard_limit * width)
        win_pos = int(window_threshold * width)

        marker = " ← HALT" if ev.halted else ""
        print(f"{ev.token:<12} {ev.coherence:>5.3f}  |{bar}|{marker}")

    print(f"{'':>19}{'|':>{int(hard_limit * width) + 1}} hard_limit={hard_limit}")
    print(f"{'':>19}{'|':>{int(window_threshold * width) + 1}} window_threshold={window_threshold}")


# Run and visualise
kernel = StreamingKernel(hard_limit=0.3, window_size=4, window_threshold=0.45)
tokens = ["The", " drug", " dosage", " is", " 200", "mg", " per", " day", " or", " 5000", "mg", " if"]
scores = [0.95, 0.92, 0.90, 0.88, 0.85, 0.80, 0.70, 0.60, 0.48, 0.30, 0.20, 0.10]

idx = 0
def sf(tok):
    global idx
    s = scores[min(idx, len(scores) - 1)]
    idx += 1
    return s

session = kernel.stream_tokens(iter(tokens), sf)
ascii_timeline(session.events, 0.3, 0.45)

## 5. False-Halt Prevention

False halts interrupt valid responses. Strategies to minimise:

| Strategy | Implementation |
|----------|----------------|
| Wider window | `window_size=8` smooths transient dips |
| Lower hard limit | `hard_limit=0.2` only catches catastrophic drops |
| Sentence-boundary scoring | Score at `.!?` instead of every token |
| Soft halt buffer | Accept N soft halts before hard stopping |

The `StreamingKernel` already implements sentence-boundary awareness —
it delays scoring until sentence ends when `sentence_boundary=True`.

## Parameter Reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `hard_limit` | 0.5 | Absolute coherence floor |
| `window_size` | 5 | Sliding window token count |
| `window_threshold` | 0.5 | Minimum window average |
| `trend_window` | 0 | Trend detection window (0=disabled) |
| `trend_threshold` | 0.3 | Max coherence drop over trend window |